In [ ]:
from pathlib import Path
import sys
code_roots=list(Path('/kaggle/input').rglob('train.json'))
code_root=next(p.parent for p in code_roots if (p.parent/'legalspan_lora.py').exists())
sys.path.insert(0,str(code_root))
models=next(p.parent for p in Path('/kaggle/input').rglob('legalspan_phase1.pt'))
teacher=next(Path('/kaggle/input').rglob('best_adapter.pt'))
print('Input code and parent found; teacher artifact:',teacher.name,flush=True)


In [ ]:
%%writefile coverage_decode.py
"""Inference-only coverage ablation for the original LegalSpan prompt route.

Coverage is source-position attention mass, not a factuality guarantee.
Reference: See et al. (2017), https://arxiv.org/abs/1704.04368 .
This pilot applies a fixed decoding penalty; it does not implement their trained
coverage feature/loss and must not be reported as that full method.
"""
import math
import types
import torch
from torch.nn import functional as F


def coverage_attention(scores, eligible, strength, previous=None, active=None):
    if strength < 0 or not math.isfinite(strength):
        raise ValueError('Coverage strength must be finite and nonnegative')
    if scores.ndim != 3 or eligible.shape != scores.shape[::2] or eligible.dtype != torch.bool:
        raise ValueError('Expected scores [B,T,S] and boolean eligibility [B,S]')
    coverage = scores.new_zeros(scores.shape[0], scores.shape[2], dtype=torch.float32) if previous is None else previous.clone()
    if coverage.shape != eligible.shape or not torch.isfinite(coverage).all() or (coverage < 0).any():
        raise ValueError('Invalid prior coverage')
    if active is None:
        active = torch.ones(scores.shape[1], dtype=torch.bool, device=scores.device)
    if active.shape != (scores.shape[1],):
        raise ValueError('Active query mask must match time dimension')
    has_source = eligible.any(-1)
    safe = eligible | ~has_source[:,None]
    rows=[]
    for t in range(scores.shape[1]):
        adjusted = scores[:,t].float() - strength * coverage if active[t] else scores[:,t].float()
        alpha = adjusted.masked_fill(~safe,-torch.inf).softmax(-1) * eligible
        rows.append(alpha)
        if active[t]:
            coverage = coverage + alpha
    return torch.stack(rows,1),coverage


def _forward(route, x, start, prompt_length, roles, eligible, counts, memory=None):
    if route.training:
        raise RuntimeError('Coverage pilot is inference-only')
    if route.cfg.route_mode != 'full' or not route.cfg.use_continuity:
        counts=torch.zeros_like(counts)
    normalized=route.norm(x)
    fresh=memory is None
    if fresh:
        prompt=normalized[:,:prompt_length]
        memory=(route.k(prompt),route.v(prompt))
        route._coverage_state=None
    key,value=memory
    query=normalized[:,start:]
    scores=torch.bmm(route.q(query).float(),key.float().transpose(1,2))/math.sqrt(route.cfg.route_dim)
    if route.cfg.route_mode=='full' and route.cfg.use_roles:
        scores=scores+torch.bmm(route.role_q(query).float(),route.role_embeddings(roles).float().transpose(1,2))/math.sqrt(route.cfg.route_dim)
    if route.cfg.route_mode=='full' and route.cfg.use_continuity:
        scores=scores+F.softplus(route.gamma.float())*torch.log1p(counts)
    active=torch.ones(query.shape[1],device=x.device,dtype=torch.bool)
    if fresh:
        active=torch.arange(start,x.shape[1],device=x.device)>=prompt_length
    alpha,route._coverage_state=coverage_attention(scores,eligible,route._coverage_strength,route._coverage_state,active)
    context=torch.bmm(alpha.to(value.dtype),value)
    continuity_strength=(alpha*counts).sum(-1,keepdim=True)
    gate=route.gate(torch.cat((query,context.to(query.dtype),continuity_strength.to(query.dtype)),-1)).sigmoid()
    if route.cfg.use_fusion:
        x=torch.cat((x[:,:start],x[:,start:]+route.out(context)*gate),1)
    return x,memory,alpha,context,continuity_strength


def install_coverage(model, strength):
    """Install without changing parameters or state_dict keys. Reset on new prompt.

    One model instance handles one document stream at a time. Native LegalSpan
    already restricts batches to one document. Zero strength preserves its route.
    """
    if strength < 0 or not math.isfinite(strength):
        raise ValueError('Invalid coverage strength')
    found=0
    for block in model.blocks:
        if hasattr(block,'route'):
            route=block.route
            route._coverage_strength=float(strength)
            route._coverage_state=None
            route.forward=types.MethodType(_forward,route)
            found+=1
    if not found: raise ValueError('No LegalSpan source routes found')
    return model


In [ ]:
%%writefile run_pilot.py
"""Paired validation-only decoding pilot. No test predictions or submission."""
import argparse
import collections
import hashlib
import json
import re
import time
from pathlib import Path
import torch
from coverage_decode import install_coverage


def words(text): return re.findall(r'\w+', text.lower())


def metrics(prediction,reference):
    tokens=words(prediction); gold=words(reference)
    pc,gc=collections.Counter(tokens),collections.Counter(gold)
    overlap=sum((pc&gc).values())
    grams=[tuple(tokens[i:i+3]) for i in range(max(0,len(tokens)-2))]
    return {'word_count':len(tokens),'unigram_overlap_f1_proxy':2*overlap/max(1,len(tokens)+len(gold)),
            'repeated_trigram_fraction':1-len(set(grams))/len(grams) if grams else 0.}


def main():
    parser=argparse.ArgumentParser()
    parser.add_argument('--input',required=True)
    parser.add_argument('--models',required=True)
    parser.add_argument('--teacher',required=True)
    parser.add_argument('--output',default='pilot_output')
    parser.add_argument('--limit',type=int,default=8)
    parser.add_argument('--max-new-tokens',type=int,default=384)
    args=parser.parse_args()
    from phase6_data import normalize_rows,split_rows,select_passages
    import legalspan_lora as native
    out=Path(args.output);out.mkdir(parents=True,exist_ok=True)
    if not torch.cuda.is_available():raise RuntimeError('Pilot requires GPU')
    model,tok,meta=native.build(args.models,'cuda');model.eval()
    adapter_path=Path(args.teacher)
    record=torch.load(adapter_path,map_location='cpu',weights_only=False)
    parent=record['parent_metadata']
    if parent['checkpoint_sha256']!=meta['checkpoint_sha256'] or parent['tokenizer_sha256']!=meta['tokenizer_sha256']:
        raise ValueError('Phase 6 adapter parent/tokenizer mismatch')
    params=dict(model.named_parameters())
    expected={n for n,p in params.items() if p.requires_grad}
    if set(record['adapter'])!=expected:raise ValueError('Incomplete or unexpected adapter state')
    with torch.no_grad():
        for n,v in record['adapter'].items():params[n].copy_(v.to(params[n].device))
    rows=normalize_rows(json.loads((Path(args.input)/'train.json').read_text()),True,drop_empty_summaries=True)
    validation=split_rows(rows)['validation'][:args.limit]
    report={'settings':vars(args),'parent':meta,'adapter_sha256':hashlib.sha256(adapter_path.read_bytes()).hexdigest(),
            'selected_epoch':record['epoch'],'validation_ids':[r['id'] for r in validation],
            'metric_note':'Unigram overlap proxy is not official ROUGE or BERTScore. Validation only; no confirmation/test selection.',
            'runs':[]}
    print(json.dumps({'event':'model_loaded','device':'cuda','parent_sha256':meta['checkpoint_sha256'],'epoch':record['epoch']}),flush=True)
    for strength in (0.,.5):
        install_coverage(model,strength)
        for row in validation:
            path=out/f"strength_{strength}_{row['id']}.json"
            if path.exists():
                result=json.loads(path.read_text());report['runs'].append(result);continue
            tokens=tok.encode(row['document'])
            source=[t for part in select_passages(tokens,8,384) for t in part]
            ids=torch.tensor([source],device='cuda');mask=torch.ones_like(ids,dtype=torch.bool)
            began=time.monotonic()
            print(json.dumps({'event':'generation_start','id':row['id'],'strength':strength,'source_tokens':len(source)}),flush=True)
            with torch.inference_mode(),torch.autocast('cuda',dtype=torch.float16):
                generated=native.generate(model,ids,mask,args.max_new_tokens,tok.eos_id)[0]
            prediction=tok.decode([x for x in generated if x not in (tok.bos_id,tok.eos_id,tok.pad_id)])
            result={'id':row['id'],'strength':strength,'summary':prediction,'reference':row['summary'],
                    'metrics':metrics(prediction,row['summary']),'elapsed_seconds':time.monotonic()-began}
            path.write_text(json.dumps(result,ensure_ascii=False,indent=2),encoding='utf-8')
            report['runs'].append(result)
            (out/'report.json').write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')
            print(json.dumps({'event':'generation_complete','id':row['id'],'strength':strength,**result['metrics'],'elapsed_seconds':result['elapsed_seconds']}),flush=True)
    print(json.dumps({'event':'pilot_complete','documents':len(validation),'comparisons':len(report['runs'])}),flush=True)

if __name__=='__main__':main()


In [ ]:
import run_pilot
sys.argv=['run_pilot.py','--input',str(code_root),'--models',str(models),'--teacher',str(teacher),'--output','/kaggle/working/coverage_pilot']
run_pilot.main()